In [0]:
customers_df=(
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("path","/Volumes/workspace/default/practice/CustomerInfo.csv")
    .load()
)
display(customer_df)

In [0]:
from pyspark.sql.functions import *

In [0]:
enddate = lit('9999-12-31').cast("timestamp")
print(enddate)

In [0]:
customersSCD_df=(
    customers_df.withColumn("StartDate",lit('2024-08-10').cast("timestamp"))
                            .withColumn("EndDate",enddate)
                            .withColumn("IsActive",lit('Y')))
display(customersSCD_df)

In [0]:
%sql
drop table if exists customers_SCD

In [0]:
customersSCD_df.write.saveAsTable("customers_SCD")

In [0]:
%sql
select * from customers_SCD

In [0]:
customerUpdate_df = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("path","/Volumes/workspace/default/practice/CustomerUpdate.csv")
    .load()
    .withColumnRenamed("CustomerKey","CustomerKey_Upd")
    .withColumnRenamed("Fullname","Fullname_Upd")
    .withColumnRenamed("EmailAddress","EmailAddress_Upd")
    .withColumnRenamed("Phone","Phone_Upd"))

In [0]:
customerHash_df = customersSCD_df.withColumn("HashKey",xxhash64("EmailAddress","Phone"))
display(customerHash_df)

In [0]:
customerUpdateHash_df = customerUpdate_df.withColumn("HashKey",xxhash64("EmailAddress_Upd","Phone_Upd"))
display(customerUpdateHash_df)

In [0]:
joined_df = customerUpdateHash_df.join(customerHash_df, customerHash_df.HashKey == customerUpdateHash_df.HashKey, "left")
display(joined_df)

In [0]:
from datetime import *

In [0]:
startdate = lit(datetime.now()+timedelta(days=1))
staged_df = joined_df.filter(col("CustomerKey").isNull().select(
    "CustomerKey_Upd",
    "Fullname_Upd",
    "EmailAddress_Upd",
    "Phone_Upd",
    "HashKey_Upd"
).withColumn("StartDate",startdate)
                             .withColumn("EndDate",enddate)
                             )
display(staged_df)

In [0]:
from delta.tables import *

In [0]:
customerDeltaTable = DeltaTable.forName(spark,"customer_SCD")

In [0]:
today = current_timestamp()
customerDeltaTable.alias("Target").merge(staged_df.alias("Source"),"Target.CustomerKey = Source.CustomerKey_Upd").whenMatched(
    condition="Target.IsActive='Y'",set={
        "Target.IsActive":lit('N'),
        "Target.EndDate":today
    }
).whenNotMatchedInsert(
    values={
        "CustomerKey":"Source.CustomerKey_Upd",
        "Fullname":"Source.Fullname_upd",
        "EmailAddress":"Source.EmailAddress_Upd",
        "Phone":"Source.Phone_Upd",
        "StartDate":"Source.StartDate",
        "EndDate":"Source.EndDate",
        "IsActive":lit('Y')
    }
).execute()

In [0]:
%sql
select * from customer_SCD

In [0]:
customerInsert_df = staged_df.join(customers_df,staged_df.customerKey_Upd == customers_df.CustomerKey)
display(customerInsert_df)

In [0]:
customerInsertFinal_df = customerInsert_df.select(
    customerInsert_df.customerKey_df.alias("CustomerKey"),
    customerInsert_df.Fullname.alias("Fullname"),
    customerInsert_df.EmailAddress.alias("EmailAddress"),
    customerInsert_df.Phone.alias("Phone"),
    customerInsert_df.StartDate,
    customerInsert_df.EndDate
).withColumn("IsActive",lit('Y'))
display(customerInsertFinal_df)

In [0]:
customerInsertFinal_df.write.mode("append").saveAsTable("customer_SCD")

In [0]:
%sql
select * from customers_SCD